# COVID (full dataset): low-data multimodal pipeline
Same method as `busi_covid_multimodal_pipeline_fast.ipynb` (frozen ViT-B/16 global feature + small-kernel local branch + MedGemma text + fusion), but on the **full** COVID-19 Radiography Database, so the training fractions mean the same thing as in the DKA paper and the `others_run` notebooks (0.63% = 106 training images).

**Plan**

| Variants | Fractions |
|---|---|
| Linear probe, global MLP | 0.63, 1.25, 12.5, 100% |
| + local | 0.63, 1.25, 12.5% |
| Text variants (+ desc, + Q&A, full, text-only, shuffled text) | 0.63% (add 0.0125 to `TEXT_FRACTIONS` if you have time) |

**What is different from the 500-per-class notebook**
- Full dataset (21,165 images); fixed stratified 20% test split, then the test set is capped at `TEST_CAP` = 400 (stratified) to keep captioning time sane.
- CLS features are extracted for every image used (small). Patch tokens, needed only by the local branch, are kept only for images that a local or text run uses, and they live in **system RAM**, not on the GPU; each batch is sent to the GPU when needed. This keeps the GPU free for llama-server while captioning.
- Outputs go to `./mm_pipeline_covid_full/`; the dataset is reused from `./Dataset_COVID/`.

**Rough time on an RX 6700 XT:** features ~5 min, visual runs ~1 h, captioning ~800 images x 11 s ~ 2.5 h, the rest ~15 min. Every step is cached and resumable: re-run from the top after any interruption.

Needs `HF_TOKEN` / `KAGGLE_API_TOKEN` in `E:\Research\.env` (Kaggle only if `Dataset_COVID` is missing), and the llama.cpp files in `E:\Research\tools` (see Cell 6). Close other heavy programs first: about 2-3 GB of RAM is used for patch features.

In [1]:
# CELL 0 - Setup and config (full COVID)
import os, re, json, time, glob, gc, random, hashlib, subprocess, warnings
import numpy as np, pandas as pd
import torch, torch.nn as nn, torch.nn.functional as F
from PIL import Image
from tqdm.auto import tqdm
warnings.filterwarnings("ignore")
import sys
IN_COLAB = "google.colab" in sys.modules

# local runs: load HF_TOKEN / KAGGLE_API_TOKEN from the nearest .env found walking up from the working dir
def _load_env():
    d = os.path.abspath(os.getcwd())
    while True:                                       # walk up to the nearest .env (no python-dotenv needed)
        f = os.path.join(d, ".env")
        if os.path.isfile(f):
            for line in open(f, encoding="utf8"):
                line = line.strip()
                if line and not line.startswith("#") and "=" in line:
                    k, v = line.split("=", 1)
                    os.environ.setdefault(k.strip().removeprefix("export ").strip(), v.strip().strip("'\""))
            return f
        if os.path.dirname(d) == d: return None
        d = os.path.dirname(d)
if not IN_COLAB:
    print(".env loaded from:", _load_env())


# ---------------- experiment config ----------------
DATASET = "COVID"
SEED = 42                                   # train/test split seed (fixed)
MAX_PER_CLASS = None                        # full dataset (21,165 images)
TEST_CAP = 400                              # stratified cap on the 20% test split (None = all 4,233 test images)
FRACTIONS = [0.0063, 0.0125, 0.125, 1.0]    # linear probe + global MLP run at all of these
LOCAL_FRACTIONS = [0.0063, 0.0125, 0.125]   # + local (needs patch tokens; 100% skipped to save RAM)
TEXT_FRACTIONS = [0.0063]                   # text variants; add 0.0125 if you have ~1.5 h more GPU time
SUBSET_SEEDS = [0, 1, 2, 3, 4]
SEEDS_FOR = {0.0063: [0, 1, 2, 3, 4], 0.0125: [0, 1, 2, 3, 4], 0.125: [0, 1, 2], 1.0: [0, 1]}
POOL_FRAC = 0.05                            # low-data subsets (<= 5% of train) are drawn from this fixed pool, so few images need captions
CAPTION_FULL_TRAIN = False
EPOCHS, BATCH = 100, 8
BIG_BATCH = 32                              # used when a train subset has >=128 images
CAP_BATCH = 1                               # images per caption request
LR_HEAD, LR_MODULES = 1e-4, 1e-3            # asynchronous LRs (same as the BUSI run)
HID, MID_DIM, K_SMALL = 128, 16, 5          # fusion width, local bottleneck, small kernel
P_DROP, P_MOD = 0.3, 0.2                    # dropout, modality dropout

MODALITY = "chest X-ray"
KAGGLE_DS = "tawsifurrahman/covid19-radiography-database"
HARD_LEAK = re.compile(r"covid|sars|coronavirus|pneumoni", re.I)
SOFT_FLAG = re.compile(r"\bnormal\b|opacit|viral", re.I)
# 10 questions derived from BioVLM's GPT-4o class attributes (attributes_bio/covid19.json in
# github.com/mainaksingha01/BioVLM; classes covid / lung opacity / normal / viral pneumonia, same as this dataset):
# each question covers one visual feature the class lists describe differently, phrased so it never names a class.
# Not used: change over time (c10) and clinical history (lo9, lo10), which one image cannot show.
QUESTIONS = [
    "Where are any opacities: unilateral or bilateral, peripheral or perihilar, upper, mid or lower zones? If there are none, say so.",  # c1 c9 v1 v6
    "Are there ground-glass opacities? If so, are they patchy, diffuse or nodular?",  # c1 v2 lo3
    "Is there consolidation, and are air bronchograms visible within it?",  # c3 c4 v5 lo8
    "Are there reticular or interstitial markings, or peribronchial cuffing?",  # c2 v3 v8
    "Describe the vascular and bronchovascular markings from the hila: normal, prominent, or obscured by opacity?",  # c5 lo2 n3 v8
    "Is there pleural effusion or blunting of the costophrenic angles?",  # c6 v7 n6 n7
    "Are the lung volumes symmetrical and preserved, or is there volume loss or displacement of adjacent structures?",  # v10 lo7 n5
    "Are the borders of any opacity well-defined or ill-defined, and do they follow anatomical segments or lobes?",  # lo4 lo6 v9 c3
    "Are the diaphragmatic and cardiac borders sharp and distinct?",  # n2 n8
    "Are there nodules or masses, and are the lung fields overall clear and well aerated?"  # n1 n9 n10 lo1
]

# ---------------- paths / environment ----------------
DRIVE_DIR = "/content/drive/MyDrive/mm_pipeline_covid_full"
try:
    from google.colab import drive, userdata
    drive.mount("/content/drive")
except Exception:
    DRIVE_DIR = "./mm_pipeline_covid_full"
    userdata = None
os.makedirs(DRIVE_DIR, exist_ok=True)
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
torch.backends.cudnn.benchmark = True
print("dataset:", DATASET, "| device:", DEVICE, "| outputs:", DRIVE_DIR)

def secret(name):
    try:
        return userdata.get(name)
    except Exception:
        return os.environ.get(name)

def set_seed(s):
    random.seed(s); np.random.seed(s); torch.manual_seed(s); torch.cuda.manual_seed_all(s)

.env loaded from: e:\Research\.env
dataset: COVID | device: cuda | outputs: ./mm_pipeline_covid_full


In [2]:
# CELL 1 - Data, labels, fixed split (+ stratified test cap)
def download_dataset():
    root = f"/content/Dataset_{DATASET}" if IN_COLAB else f"./Dataset_{DATASET}"
    if os.path.isdir(root):
        return root
    tok = secret("KAGGLE_API_TOKEN")
    assert tok, "Add KAGGLE_API_TOKEN to Colab Secrets (and revoke any token you pasted in a notebook)."
    os.makedirs(os.path.expanduser("~/.kaggle"), exist_ok=True)
    p = os.path.expanduser("~/.kaggle/access_token")
    open(p, "w").write(tok); os.chmod(p, 0o600)
    os.environ["KAGGLE_API_TOKEN"] = tok
    from kaggle.api.kaggle_api_extended import KaggleApi   # in-process: no dependency on the kaggle executable being on PATH
    api = KaggleApi(); api.authenticate()
    api.dataset_download_files(KAGGLE_DS, path=root, unzip=True, quiet=False)
    return root

def load_dataset():
    root = download_dataset()
    paths = sorted(glob.glob(f"{root}/**/*.png", recursive=True) + glob.glob(f"{root}/**/*.jpg", recursive=True))
    paths = [p for p in paths if "mask" not in p.lower()]   # drop segmentation masks
    def _label(p):   # COVID layout is <Class>/images/<file>.png, BUSI is <class>/<file>.png
        d = os.path.basename(os.path.dirname(p))
        return os.path.basename(os.path.dirname(os.path.dirname(p))) if d.lower() == "images" else d
    d = pd.DataFrame({"image_path": paths, "label": [_label(p) for p in paths]})
    if MAX_PER_CLASS:   # keeps Colab RAM / captioning time manageable (see config)
        d = pd.concat([g.sample(min(len(g), MAX_PER_CLASS), random_state=SEED) for _, g in d.groupby("label")])
        d = d.sort_values("image_path").reset_index(drop=True)
    return d

from sklearn.model_selection import train_test_split
SPLIT_CSV = f"{DRIVE_DIR}/split.csv"
raw = load_dataset()
if os.path.exists(SPLIT_CSV):
    df = pd.read_csv(SPLIT_CSV)
else:
    tr_, te_ = train_test_split(raw, test_size=0.2, stratify=raw["label"], random_state=SEED)
    if TEST_CAP and len(te_) > TEST_CAP:     # keep a stratified TEST_CAP-image test set; the rest of the 20% is never used
        te_, _ = train_test_split(te_, train_size=TEST_CAP, stratify=te_["label"], random_state=SEED)
    df = pd.concat([tr_.assign(split="train"), te_.assign(split="test")], ignore_index=True)
    df.to_csv(SPLIT_CSV, index=False)
df["id"] = df["image_path"].apply(lambda p: hashlib.md5(p.encode()).hexdigest()[:10] + "_" + os.path.basename(p))
CLASSES = sorted(df["label"].unique())
df["y"] = df["label"].map({c: i for i, c in enumerate(CLASSES)})
NCLS, N = len(CLASSES), len(df)
train_idx_all = np.where(df["split"].values == "train")[0]
test_idx = np.where(df["split"].values == "test")[0]
print(CLASSES, "| total", N, "| train", len(train_idx_all), "| test", len(test_idx))
print(df.groupby(["split", "label"]).size().unstack())

['COVID', 'Lung_Opacity', 'Normal', 'Viral Pneumonia'] | total 17332 | train 16932 | test 400
label  COVID  Lung_Opacity  Normal  Viral Pneumonia
split                                              
test      68           114     193               25
train   2893          4809    8154             1076


In [3]:
# CELL 2 - Class-balanced, nested low-data subsets (5-fold CV equivalents)
# Caption pool: a class-stratified random POOL_FRAC of the training set (seeded). Subsets with frac <= POOL_FRAC are
# drawn from it, so every subset is still a random class-balanced sample of the train set, but only ~POOL_FRAC of
# train ever needs captions. Fractions above POOL_FRAC (e.g. 100%) use the whole train set.
_r = np.random.RandomState(SEED); _pool = []
for c in range(NCLS):
    full = train_idx_all[df["y"].values[train_idx_all] == c]
    _pool.append(_r.permutation(full)[:max(1, int(round(POOL_FRAC * len(full))))])
pool_idx = np.concatenate(_pool)

def subset(frac, seed):
    rng = np.random.RandomState(seed); out = []
    for c in range(NCLS):
        full = train_idx_all[df["y"].values[train_idx_all] == c]
        n_c = max(1, int(round(frac * len(full))))
        src_ids = pool_idx[df["y"].values[pool_idx] == c] if frac <= POOL_FRAC else full
        src_ids = src_ids[rng.permutation(len(src_ids))]
        out.append(src_ids[:min(n_c, len(src_ids))])
    return np.concatenate(out)

for f in FRACTIONS:
    print(f"{f*100:g}% ->", len(subset(f, 0)), "train images (seed 0)")
# Images that need patch tokens: everything a local-branch or text run touches (+ the test set).
_need = set(test_idx.tolist())
for f in sorted(set(LOCAL_FRACTIONS) | set(TEXT_FRACTIONS)):
    for s in SEEDS_FOR.get(round(f, 4), SUBSET_SEEDS):
        _need |= set(subset(f, s).tolist())
PATCH_ROWS = np.array(sorted(_need))
print(f"patch tokens kept for {len(PATCH_ROWS)} of {N} images (~{len(PATCH_ROWS) * 196 * 768 * 2 / 1e9:.1f} GB of system RAM)")


0.63% -> 106 train images (seed 0)
1.25% -> 211 train images (seed 0)
12.5% -> 2116 train images (seed 0)
100% -> 16932 train images (seed 0)
patch tokens kept for 6425 of 17332 images (~1.9 GB of system RAM)


In [4]:
# CELL 3 - Frozen ViT-B/16 features: CLS for every image, patch tokens only for PATCH_ROWS (kept in system RAM)
import timm
from timm.data import resolve_data_config, create_transform
FEATS = f"{DRIVE_DIR}/vit_feats.pt"
ids_all = df["id"].tolist(); ids_patch = [ids_all[i] for i in PATCH_ROWS]
blob = torch.load(FEATS) if os.path.exists(FEATS) else None
if blob is not None and blob["ids"] == ids_all and blob["patch_ids"] == ids_patch:
    cls_all, patch_sel = blob["cls"], blob["patch"]
    print("loaded cached features")
else:
    if blob is not None: print("cached features do not match the current split/fractions: recomputing")
    del blob; gc.collect()
    vit = timm.create_model("vit_base_patch16_224", pretrained=True).to(DEVICE).eval()
    tfm = create_transform(**resolve_data_config({}, model=vit))
    class DS(torch.utils.data.Dataset):
        def __len__(s): return N
        def __getitem__(s, i): return tfm(Image.open(df["image_path"][i]).convert("RGB"))
    prow_np = np.full(N, -1, dtype=np.int64); prow_np[PATCH_ROWS] = np.arange(len(PATCH_ROWS))
    cls_all = torch.empty(N, 768, dtype=torch.float16)
    patch_sel = torch.empty(len(PATCH_ROWS), 196, 768, dtype=torch.float16)
    loader = torch.utils.data.DataLoader(DS(), batch_size=64, num_workers=0 if os.name == "nt" else 2)
    k = 0
    with torch.no_grad(), torch.autocast(DEVICE, dtype=torch.float16, enabled=DEVICE == "cuda"):
        for x in tqdm(loader, desc="ViT features"):
            t = vit.forward_features(x.to(DEVICE)).half().cpu()          # B,197,768
            rows = np.arange(k, k + len(t)); cls_all[rows] = t[:, 0]
            keep = prow_np[rows] >= 0
            if keep.any(): patch_sel[prow_np[rows][keep]] = t[torch.as_tensor(keep), 1:]
            k += len(t)
    torch.save({"ids": ids_all, "patch_ids": ids_patch, "cls": cls_all, "patch": patch_sel}, FEATS)
    del vit
    gc.collect(); torch.cuda.empty_cache()
print("cls", tuple(cls_all.shape), "| patch", tuple(patch_sel.shape))

PROW = torch.full((N,), -1, dtype=torch.long); PROW[torch.as_tensor(PATCH_ROWS)] = torch.arange(len(PATCH_ROWS))
try: patch_sel = patch_sel.pin_memory()       # faster CPU->GPU copies; skipped if not supported
except Exception: pass
data = {
    "G": cls_all.to(DEVICE), "P": patch_sel, "PROW": PROW,                 # P stays in system RAM
    "D": torch.zeros(N, 512, device=DEVICE), "Q": torch.zeros(N, 512, device=DEVICE),
    "y": torch.as_tensor(df["y"].values, device=DEVICE).long(),
    "has_text": np.zeros(N, dtype=bool),
}


ViT features:   0%|          | 0/271 [00:00<?, ?it/s]

cls (17332, 768) | patch (6425, 196, 768)


In [5]:
# CELL 4 - Models and training engine
from sklearn.metrics import accuracy_score, f1_score, recall_score

class LocalBranch(nn.Module):
    def __init__(s, d=768, mid=MID_DIM, out=HID, k=K_SMALL):
        super().__init__()
        s.down = nn.Linear(d, mid)
        s.dw = nn.Conv2d(mid, mid, k, padding=k // 2, groups=mid)
        s.act = nn.GELU(); s.up = nn.Linear(mid, out)
    def forward(s, p):
        x = s.down(p).transpose(1, 2).reshape(p.size(0), -1, 14, 14)
        return s.up(s.act(s.dw(x)).mean((2, 3)))

class LinearProbe(nn.Module):
    def __init__(s, n_cls):
        super().__init__(); s.fc = nn.Linear(768, n_cls)
    def forward(s, b): return s.fc(b["g"])

class Fusion(nn.Module):
    SRC = {"g": "g", "l": "p", "d": "d", "q": "q"}
    def __init__(s, n_cls, g=0, l=0, d=0, q=0):
        super().__init__()
        s.enc = nn.ModuleDict()
        if g: s.enc["g"] = nn.Sequential(nn.LayerNorm(768), nn.Linear(768, HID))
        if l: s.enc["l"] = LocalBranch()
        if d: s.enc["d"] = nn.Sequential(nn.LayerNorm(512), nn.Linear(512, HID))
        if q: s.enc["q"] = nn.Sequential(nn.LayerNorm(512), nn.Linear(512, HID))
        s.keys = list(s.enc.keys())
        s.head = nn.Sequential(nn.Dropout(P_DROP), nn.Linear(len(s.keys) * HID, HID), nn.GELU(), nn.Linear(HID, n_cls))
    def forward(s, b):
        z = torch.stack([s.enc[k](b[s.SRC[k]]) for k in s.keys], 1)
        if s.training and z.size(1) > 1 and P_MOD > 0:
            m = (torch.rand(z.size(0), z.size(1), device=z.device) > P_MOD).float()
            none = m.sum(1) == 0
            if none.any():
                r = torch.randint(0, z.size(1), (int(none.sum()),), device=z.device)
                m[none.nonzero().squeeze(1), r] = 1
            z = z * m.unsqueeze(-1)
        return s.head(z.flatten(1))

VARIANTS = {
    "linear_probe":       dict(kind="linear"),
    "global_mlp":         dict(g=1),
    "global+local":       dict(g=1, l=1),
    "global+desc":        dict(g=1, d=1),
    "global+qa":          dict(g=1, q=1),
    "full":               dict(g=1, l=1, d=1, q=1),
    "text_only":          dict(d=1, q=1),
    "full_shuffled_text": dict(g=1, l=1, d=1, q=1, shuffle=True),
}
needs_text = lambda spec: bool(spec.get("d") or spec.get("q"))

def make_batch(idx, tidx, need_p=False):
    """idx / tidx are CPU index tensors. Patch tokens come from system RAM only when the model has a local branch."""
    i, t = idx.to(DEVICE), tidx.to(DEVICE)
    b = {"g": data["G"][i].float(), "d": data["D"][t], "q": data["Q"][t]}
    if need_p:
        r = data["PROW"][idx]
        assert (r >= 0).all(), "patch tokens missing for some images: add this fraction to LOCAL_FRACTIONS/TEXT_FRACTIONS and re-run Cells 2-3"
        b["p"] = data["P"][r].to(DEVICE, non_blocking=True).float()
    return b

def train_eval(spec, tr_idx, te_idx, seed):
    set_seed(seed)
    model = (LinearProbe(NCLS) if spec.get("kind") == "linear"
             else Fusion(NCLS, **{k: spec.get(k, 0) for k in "glqd"})).to(DEVICE)
    if spec.get("kind") == "linear":
        groups = [{"params": model.parameters(), "lr": LR_HEAD}]
    else:
        head = list(model.head[-1].parameters()); hid = {id(p) for p in head}
        groups = [{"params": head, "lr": LR_HEAD},
                  {"params": [p for p in model.parameters() if id(p) not in hid], "lr": LR_MODULES}]
    opt = torch.optim.Adam(groups)
    tr_t, te_t = tr_idx.copy(), te_idx.copy()
    if spec.get("shuffle"):   # control: every image gets the text of a DIFFERENT, randomly chosen image
        rng = np.random.RandomState(seed + 999)
        def derange(a):
            if len(a) < 2: return a
            while True:
                p = a[rng.permutation(len(a))]
                if (p != a).all(): return p
        tr_t, te_t = derange(tr_t), derange(te_t)
    need_p = bool(spec.get("l"))
    tr_i, tr_ti = torch.as_tensor(tr_idx), torch.as_tensor(tr_t)                 # CPU indices
    te_i, te_ti = torch.as_tensor(te_idx), torch.as_tensor(te_t)
    n = len(tr_idx); bs = BIG_BATCH if n >= 128 else BATCH
    for ep in range(EPOCHS):
        model.train(); perm = torch.randperm(n)
        for i in range(0, n, bs):
            bi = perm[i:i + bs]
            loss = F.cross_entropy(model(make_batch(tr_i[bi], tr_ti[bi], need_p)), data["y"][tr_i[bi].to(DEVICE)])
            opt.zero_grad(); loss.backward(); opt.step()
    model.eval(); logits = []
    with torch.no_grad():
        for i in range(0, len(te_i), 256):
            logits.append(model(make_batch(te_i[i:i + 256], te_ti[i:i + 256], need_p)).float().cpu())
    prob = torch.cat(logits).softmax(1).numpy(); pred = prob.argmax(1)
    yt = data["y"][te_i.to(DEVICE)].cpu().numpy()
    return dict(acc=100 * accuracy_score(yt, pred), f1=f1_score(yt, pred, average="macro"),
                sen=100 * recall_score(yt, pred, average="macro"),
                params=sum(p.numel() for p in model.parameters() if p.requires_grad)), prob

RESULTS_CSV = f"{DRIVE_DIR}/results_v2.csv"     # v2 = same training, plus per-image predictions; v1 stays in results_fast.csv
COLS = ["variant", "frac", "seed", "n_train", "acc", "f1", "sen", "params", "text_tag"]
# text_tag = the caption file a text run used. A new prompt/question set gets a new caption file, so its text
# runs are trained fresh instead of being skipped as "done"; visual-only runs have an empty tag and are reused.
text_tag = lambda spec: os.path.basename(CAPS) if needs_text(spec) else ""

PREDS_CSV = f"{DRIVE_DIR}/preds_v2.csv"          # one row per test image per run (for McNemar tests and the H8 analysis)
def save_preds(nm, f, s, tag, prob):
    out = pd.DataFrame({"variant": nm, "frac": round(f, 4), "seed": s, "text_tag": tag,
                        "image_id": df["id"].values[test_idx], "y_true": df["y"].values[test_idx], "y_pred": prob.argmax(1)})
    for k in range(prob.shape[1]): out[f"p{k}"] = prob[:, k].round(5)
    out.to_csv(PREDS_CSV, mode="a", header=not os.path.exists(PREDS_CSV), index=False)

def load_results():
    if not os.path.exists(RESULTS_CSV): return pd.DataFrame(columns=COLS)
    res = pd.read_csv(RESULTS_CSV)
    if "text_tag" not in res:                      # results from before tagging: their text runs used the old questions
        res["text_tag"] = np.where(res["variant"].map(lambda v: needs_text(VARIANTS.get(v, {}))), "legacy", "")
        res[COLS].to_csv(RESULTS_CSV, index=False)
    res["text_tag"] = res["text_tag"].fillna("")
    return res

def run_grid(names, fractions=FRACTIONS, seeds=None):
    res = load_results()
    done = set(zip(res["variant"], res["frac"].round(4), res["seed"], res["text_tag"]))
    jobs = [(nm, f, s) for nm in names for f in fractions
            for s in (seeds or SEEDS_FOR.get(round(f, 4), SUBSET_SEEDS))
            if (nm, round(f, 4), s, text_tag(VARIANTS[nm])) not in done]
    for nm, f, s in tqdm(jobs):
        spec, tr = VARIANTS[nm], subset(f, s)
        if needs_text(spec) and not (data["has_text"][tr].all() and data["has_text"][test_idx].all()):
            print(f"skip {nm} {f*100:g}% seed {s}: captions missing for needed images"); continue
        m, prob = train_eval(spec, tr, test_idx, s)
        save_preds(nm, f, s, text_tag(spec), prob)                    # predictions first, then the results row
        row = dict(variant=nm, frac=round(f, 4), seed=s, n_train=len(tr), **m, text_tag=text_tag(spec))
        pd.DataFrame([row])[COLS].to_csv(RESULTS_CSV, mode="a", header=not os.path.exists(RESULTS_CSV), index=False)  # resumable
    return load_results()

In [6]:
# CELL 5 - Phases 1-2: baselines (all fractions incl. 100%) and local branch (no 100%)
yt_ = df["y"].values[test_idx]
print(f"test set: {len(test_idx)} images | majority-class accuracy: {100 * np.bincount(yt_.astype(int)).max() / len(yt_):.1f}%")
run_grid(["linear_probe", "global_mlp"], fractions=FRACTIONS)
res = run_grid(["global+local"], fractions=LOCAL_FRACTIONS)
print(res.groupby(["variant", "frac"])[["acc", "f1", "sen"]].mean().round(2))


test set: 400 images | majority-class accuracy: 48.2%


  0%|          | 0/30 [00:00<?, ?it/s]

  0%|          | 0/13 [00:00<?, ?it/s]

                       acc    f1    sen
variant      frac                      
global+local 0.0063  74.05  0.70  67.25
             0.0125  77.60  0.74  71.94
             0.1250  86.67  0.87  85.33
global_mlp   0.0063  74.20  0.71  67.63
             0.0125  79.50  0.77  73.92
             0.1250  85.00  0.85  84.32
             1.0000  90.88  0.92  90.84
linear_probe 0.0063  72.00  0.62  57.77
             0.0125  74.40  0.63  58.85
             0.1250  85.58  0.84  82.44
             1.0000  89.62  0.90  88.32


In [7]:
# CELL 6 - Phase 3a: caption with MedGemma (resumable)
# Two ways to run the SAME model (MedGemma 4B-it) with the SAME prompt:
#   "llamacpp"     : local llama.cpp server, Q8_0 GGUF on the GPU via Vulkan (~11 s/image on an RX 6700 XT).
#                    The reply is forced to match a JSON schema, so it always parses.
#   "transformers" : Hugging Face transformers (Colab / NVIDIA): bf16, or 4-bit with bitsandbytes.
# Each backend writes its own caption file, so captions from the two are never mixed.
import urllib.request, base64, atexit

MID = "google/medgemma-4b-it"
BUDGET_MIN = 240          # per-session time budget (~800 images x 11 s ~ 2.5 h); re-run this cell to continue
SYSTEM = "You are an expert radiologist."

TOOLS = os.path.abspath(os.path.join("..", "tools"))         # E:\Research\tools (not in git)
LLAMA_SERVER = os.path.join(TOOLS, "llama.cpp-b12383", "llama-server.exe")
GGUF_MODEL = os.path.join(TOOLS, "gguf", "medgemma-4b-it-Q8_0.gguf")
GGUF_MMPROJ = os.path.join(TOOLS, "gguf", "mmproj-F16.gguf")
LLAMA_PORT = 8089
BACKEND = "llamacpp" if (not IN_COLAB and all(map(os.path.exists, [LLAMA_SERVER, GGUF_MODEL, GGUF_MMPROJ]))) else "transformers"

def build_prompt():
    qs = "\n".join(f"{i+1}. {q}" for i, q in enumerate(QUESTIONS))
    return (f"This is a {MODALITY} image.\n"
            "Return ONLY valid JSON, nothing else, with exactly this structure:\n"
            '{"description": "<2-3 short sentences describing the visible findings>", '
            f'"answers": ["<answer 1>", ..., "<answer {len(QUESTIONS)}>"]}}\n'
            f"The list must contain exactly {len(QUESTIONS)} strings, one per question, each AT MOST 15 words:\n{qs}\n"
            "Do NOT state a diagnosis, disease name, or BI-RADS category.")
PROMPT = build_prompt()
# captions live in a file named after a hash of (model/backend + prompt): a changed prompt or backend never mixes with old captions
CAP_TAG = MID if BACKEND == "transformers" else f"llamacpp:{os.path.basename(GGUF_MODEL)}+{os.path.basename(GGUF_MMPROJ)}"
CAPS = f"{DRIVE_DIR}/captions_{hashlib.md5((CAP_TAG + PROMPT).encode()).hexdigest()[:8]}.jsonl"
print("backend:", BACKEND, "| caption file:", CAPS)

if BACKEND == "llamacpp":
    LLAMA_URL = f"http://127.0.0.1:{LLAMA_PORT}"
    QUANT = "Q8_0 GGUF + F16 mmproj (unsloth/medgemma-4b-it-GGUF), llama.cpp b12383, Vulkan"
    SCHEMA = {"type": "object", "required": ["description", "answers"],
              "properties": {"description": {"type": "string"},
                             "answers": {"type": "array", "items": {"type": "string"},
                                         "minItems": len(QUESTIONS), "maxItems": len(QUESTIONS)}}}

    def _llama_ok():
        try: return json.load(urllib.request.urlopen(LLAMA_URL + "/health", timeout=2)).get("status") == "ok"
        except Exception: return False

    def start_llama():
        global _llama_proc
        if _llama_ok():
            print("llama-server already running"); return
        log = open(f"{DRIVE_DIR}/llama_server.log", "w")
        _llama_proc = subprocess.Popen([LLAMA_SERVER, "-m", GGUF_MODEL, "--mmproj", GGUF_MMPROJ, "-ngl", "99",
                                        "-c", "4096", "-fa", "on", "--host", "127.0.0.1", "--port", str(LLAMA_PORT)],
                                       stdout=log, stderr=subprocess.STDOUT)
        for _ in range(240):
            if _llama_ok():
                print("llama-server ready"); return
            if _llama_proc.poll() is not None:
                raise RuntimeError(f"llama-server exited early; see {DRIVE_DIR}/llama_server.log")
            time.sleep(1)
        raise RuntimeError(f"llama-server not ready after 4 min; see {DRIVE_DIR}/llama_server.log")

    def stop_llama():                                       # frees the GPU memory (~6 GB)
        p = globals().get("_llama_proc")
        if p is not None and p.poll() is None:
            p.terminate(); p.wait(30); print("llama-server stopped")

    _llama_proc = None
    atexit.register(stop_llama)

    def _ask_one(path, max_new_tokens, sample, seed):
        img = base64.b64encode(open(path, "rb").read()).decode()
        mime = "image/png" if path.lower().endswith(".png") else "image/jpeg"
        body = {"messages": [{"role": "system", "content": SYSTEM},
                             {"role": "user", "content": [
                                 {"type": "image_url", "image_url": {"url": f"data:{mime};base64,{img}"}},
                                 {"type": "text", "text": PROMPT}]}],
                "max_tokens": max_new_tokens, "cache_prompt": False, "seed": seed,
                "response_format": {"type": "json_schema", "json_schema": {"name": "caption", "schema": SCHEMA}}}
        body.update(dict(temperature=0.7, top_p=0.9) if sample else dict(temperature=0))
        req = urllib.request.Request(LLAMA_URL + "/v1/chat/completions", json.dumps(body).encode(),
                                     {"Content-Type": "application/json"})
        return json.load(urllib.request.urlopen(req, timeout=600))["choices"][0]["message"]["content"]

    def ask_batch(paths, max_new_tokens=450, sample=False, seed=0):
        return [_ask_one(p, max_new_tokens, sample, seed) for p in paths]

else:
    from transformers import AutoProcessor, AutoModelForImageTextToText, BitsAndBytesConfig
    hf = secret("HF_TOKEN")
    assert hf, "HF_TOKEN is empty: put HF_TOKEN=... in your .env (or Colab Secrets) and re-run Cell 0"
    os.environ["HF_TOKEN"] = hf
    proc = AutoProcessor.from_pretrained(MID, token=hf)
    proc.tokenizer.padding_side = "left"                      # required for batched generation
    try:
        import accelerate; _kw = dict(device_map="auto", token=hf)
    except ImportError:                                       # no accelerate: load on CPU, then move to the GPU below
        print("accelerate not installed (pip install accelerate); loading without device_map")
        _kw = dict(token=hf)
    _use_bnb = False
    if torch.cuda.is_available() and not getattr(torch.version, "hip", None):   # bitsandbytes 4-bit: NVIDIA only
        try:
            import bitsandbytes; _use_bnb = True
        except ImportError:
            pass
    if _use_bnb:
        _kw["quantization_config"] = BitsAndBytesConfig(load_in_4bit=True, bnb_4bit_compute_dtype=torch.float16)
    else:                                                     # ROCm / CPU: plain bf16 (~8 GB for 4B); fp16 overflows
        _kw["dtype"] = torch.bfloat16
    QUANT = "bnb 4-bit" if _use_bnb else "bf16"
    print("MedGemma load:", QUANT, "| device:", DEVICE)
    mg = AutoModelForImageTextToText.from_pretrained(MID, **_kw).eval()
    if "device_map" not in _kw: mg = mg.to(DEVICE)
    # dtype of the unquantized vision tower, so pixel_values match whatever transformers version is installed
    VDTYPE = next((p.dtype for n, p in mg.named_parameters() if "vision" in n and p.is_floating_point()), torch.float16)

    def _msgs(path):
        img = Image.open(path).convert("RGB")
        return [{"role": "system", "content": [{"type": "text", "text": SYSTEM}]},
                {"role": "user", "content": [{"type": "image", "image": img}, {"type": "text", "text": PROMPT}]}]

    # The answer is pre-filled with the start of the JSON, so the model must answer in JSON straight away
    # (otherwise MedGemma 1.5 may write a long hidden "thought" first and run out of tokens).
    PREFILL = '{"description": "'

    @torch.inference_mode()
    def ask_batch(paths, max_new_tokens=450, sample=False, seed=0):
        msgs = [_msgs(p) for p in paths]
        texts = [proc.apply_chat_template(m, add_generation_prompt=True, tokenize=False) + PREFILL for m in msgs]
        imgs = [[m[-1]["content"][0]["image"]] for m in msgs]
        inp = proc(text=texts, images=imgs, return_tensors="pt", padding=True).to(mg.device)
        if "pixel_values" in inp: inp["pixel_values"] = inp["pixel_values"].to(VDTYPE)
        n = inp["input_ids"].shape[-1]
        if sample: torch.manual_seed(seed)
        kw = dict(do_sample=True, temperature=0.7, top_p=0.9) if sample else dict(do_sample=False)
        out = mg.generate(**inp, max_new_tokens=max_new_tokens, **kw)
        return [PREFILL + proc.decode(o[n:], skip_special_tokens=True) for o in out]

def parse_json(txt):
    txt = txt.split("<unused95>")[-1]                     # drop a "thinking" block if one slipped through
    t = re.sub(r"^```(?:json)?|```$", "", txt.strip(), flags=re.M).strip()
    d = json.loads(t[t.find("{"): t.rfind("}") + 1])
    ans = [str(a) for a in d["answers"]]
    assert len(ans) == len(QUESTIONS)
    return {"description": str(d["description"]), "answers": ans}

def to_record(raw, a=0):
    try: p = parse_json(raw)
    except Exception: return dict(raw=raw, parse_ok=False, attempts=a + 1)
    # leaks are only FLAGGED here; the hard-leak words are redacted before embedding (Cell 8), so no costly regeneration
    return dict(raw=raw, parse_ok=True, hard_leak=bool(HARD_LEAK.search(json.dumps(p))), attempts=a + 1, **p)

# queue: test set first, then exactly the train images that the text experiments will use
queue, seen = list(test_idx), set(test_idx)
for f in TEXT_FRACTIONS:
    for s in SEEDS_FOR.get(round(f, 4), SUBSET_SEEDS):
        for i in subset(f, s):
            if i not in seen: queue.append(i); seen.add(i)
if CAPTION_FULL_TRAIN:
    queue += [i for i in train_idx_all if i not in seen]

done = set()
if os.path.exists(CAPS):
    done = {r["id"] for r in map(json.loads, open(CAPS)) if r.get("parse_ok")}   # failed ones get retried
todo = [i for i in queue if df["id"][i] not in done]
print(f"{len(queue)} images needed | {len(done)} done | {len(todo)} to go")
if BACKEND == "llamacpp" and todo: start_llama()     # only start the server when there is work to do
t0 = time.time()
try:
    with open(CAPS, "a") as fh:
        for k in tqdm(range(0, len(todo), CAP_BATCH)):
            if time.time() - t0 > BUDGET_MIN * 60:
                print("time budget reached; re-run this cell to continue"); break
            chunk = todo[k:k + CAP_BATCH]; paths = [df["image_path"][i] for i in chunk]
            try:
                raws = ask_batch(paths)
            except Exception as e:                            # e.g. OOM: fall back to one image at a time
                print("batch failed, going image by image:", repr(e)[:120]); torch.cuda.empty_cache(); raws = []
                for p in paths:
                    try: raws.append(ask_batch([p])[0])
                    except Exception as e2: print("  failed:", os.path.basename(p), repr(e2)[:120]); raws.append("")
            for i, raw in zip(chunk, raws):
                rec, a = to_record(raw), 0
                while not rec["parse_ok"] and a < 2:           # retry ONLY unparseable outputs, sampled
                    a += 1
                    try: rec = to_record(ask_batch([df["image_path"][i]], sample=True, seed=a)[0], a)
                    except Exception: break
                fh.write(json.dumps({"id": df["id"][i], "label": df["label"][i], **rec}) + "\n"); fh.flush()
finally:
    if BACKEND == "llamacpp": stop_llama()                 # also runs if you interrupt the cell

json.dump({"model": MID, "backend": BACKEND, "quantization": QUANT, "prompt": PROMPT, "system": SYSTEM,
           "questions": QUESTIONS, "caps_file": CAPS, "pool_frac": POOL_FRAC,
           "text_fractions": TEXT_FRACTIONS, "cap_batch": CAP_BATCH},
          open(f"{DRIVE_DIR}/run_info.json", "w"), indent=2)

backend: llamacpp | caption file: ./mm_pipeline_covid_full/captions_5d6d7f4d.jsonl
799 images needed | 0 done | 799 to go
llama-server ready


  0%|          | 0/799 [00:00<?, ?it/s]

llama-server stopped


In [8]:
# CELL 7 - Phase 3b: audit
rows = {r["id"]: r for r in map(json.loads, open(CAPS))}
cap = pd.DataFrame.from_dict(rows, orient="index")
print("captioned:", len(cap), "| parse failures:", int((~cap["parse_ok"]).sum()))
ok = cap[cap["parse_ok"]].copy()
alltext = ok.apply(lambda r: r["description"] + " " + " ".join(r["answers"]), axis=1)
ok["hard"] = alltext.apply(lambda t: bool(HARD_LEAK.search(t)))
ok["soft"] = alltext.apply(lambda t: bool(SOFT_FLAG.search(t)))
print("\nleak flags by class (before redaction):"); print(ok.groupby("label")[["hard", "soft"]].sum())

def redact(t): return re.sub(r"\s+", " ", HARD_LEAK.sub("", t)).strip()
ok["clean"] = alltext.apply(redact)

from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import make_pipeline
split_of = df.set_index("id")["split"]
trn, tst = ok[ok.index.map(split_of) == "train"], ok[ok.index.map(split_of) == "test"]
if len(trn) > 30 and len(tst) > 10:
    pipe = make_pipeline(TfidfVectorizer(ngram_range=(1, 2), min_df=2), LogisticRegression(max_iter=2000))
    pipe.fit(trn["clean"], trn["label"])
    print(f"\ntext-only TF-IDF accuracy: {100*pipe.score(tst['clean'], tst['label']):.1f}%  |  "
          f"majority baseline: {100*tst['label'].value_counts(normalize=True).iloc[0]:.1f}%  (>95% => suspect leakage)")

aud = ok.sample(min(50, len(ok)), random_state=0)[["label", "description"]].copy()
aud.insert(0, "image_path", [df.set_index("id").loc[i, "image_path"] for i in aud.index])
aud["rating_1to5"] = ""; aud["notes"] = ""
aud.to_csv(f"{DRIVE_DIR}/caption_audit_sheet.csv")

captioned: 799 | parse failures: 0

leak flags by class (before redaction):
                 hard  soft
label                      
COVID               3   130
Lung_Opacity        9   231
Normal              2   387
Viral Pneumonia     5    51

text-only TF-IDF accuracy: 74.5%  |  majority baseline: 48.2%  (>95% => suspect leakage)


In [9]:
# CELL 8 - Phase 3c: embed text (BiomedCLIP)
if "stop_llama" in globals(): stop_llama()          # free the GPU if Cell 6 left llama-server running
for _n in ("mg", "proc"): globals().pop(_n, None)
gc.collect(); torch.cuda.empty_cache()
import open_clip
BIOMED = "hf-hub:microsoft/BiomedCLIP-PubMedBERT_256-vit_base_patch16_224"
bclip, _ = open_clip.create_model_from_pretrained(BIOMED)
btok = open_clip.get_tokenizer(BIOMED)
bclip = bclip.to(DEVICE).eval()

@torch.no_grad()
def embed(texts, bs=32):
    out = []
    for i in range(0, len(texts), bs):
        t = btok(texts[i:i + bs], context_length=256).to(DEVICE)
        out.append(F.normalize(bclip.encode_text(t).float(), dim=-1).cpu())
    return torch.cat(out)

ids = list(ok.index)
desc_e = embed([redact(ok.loc[i, "description"]) for i in ids])
flat = [redact(a) for i in ids for a in ok.loc[i, "answers"]]
qa_e = embed(flat).reshape(len(ids), len(QUESTIONS), -1).mean(1)
pos = {i: k for k, i in enumerate(df["id"])}
for k, i in enumerate(ids):
    j = pos[i]
    data["D"][j] = desc_e[k].to(DEVICE); data["Q"][j] = F.normalize(qa_e[k], dim=0).to(DEVICE)
    data["has_text"][j] = True
del bclip; gc.collect(); torch.cuda.empty_cache()

In [10]:
# CELL 9 - Phase 4: Fusion runs
res = run_grid(["global+desc", "global+qa", "full", "text_only", "full_shuffled_text"], fractions=TEXT_FRACTIONS)

  0%|          | 0/25 [00:00<?, ?it/s]

In [11]:
# CELL 10 - Phase 5: results table & figure
import matplotlib; matplotlib.use("Agg")
import matplotlib.pyplot as plt
res = load_results()
_cur = os.path.basename(CAPS) if "CAPS" in globals() else None   # report text runs from the current captions only
res = res[(res["text_tag"] == "") | (res["text_tag"] == _cur)]
print("text results from:", _cur)
order = [v for v in VARIANTS if v in set(res["variant"])]
fr = sorted(res["frac"].unique())
for metric in ["acc", "f1", "sen"]:
    g = res.groupby(["variant", "frac"])[metric].agg(["mean", "std"])
    tab = g.apply(lambda r: f"{r['mean']:.2f}±{r['std']:.2f}" if metric != "f1" else f"{r['mean']:.3f}±{r['std']:.3f}", axis=1).unstack("frac")
    tab = tab.reindex(order)[fr]; tab.columns = [f"{c*100:g}%" for c in fr]
    print(f"\n=== {metric.upper()} (COVID full) ===")
    print(tab.to_string()); tab.to_csv(f"{DRIVE_DIR}/table_{metric}.csv")

fig, ax = plt.subplots(figsize=(8, 5))
for v in order:
    s = res[res.variant == v].groupby("frac")["acc"].agg(["mean", "std"]).reindex(fr)
    ax.errorbar(range(len(fr)), s["mean"], yerr=s["std"], marker="o", capsize=2, label=v,
                ls="--" if "shuffled" in v or v == "linear_probe" else "-")
ax.set_xticks(range(len(fr))); ax.set_xticklabels([f"{c*100:g}%" for c in fr])
ax.set_xlabel("Training set (%)"); ax.set_ylabel("Test ACC (%)")
ax.set_title("COVID (full dataset): low-data multimodal variants")
ax.legend(fontsize=8); ax.grid(alpha=.3); fig.tight_layout(); fig.savefig(f"{DRIVE_DIR}/acc_vs_fraction.png", dpi=200)


=== ACC (COVID full) ===
                         0.63%       1.25%       12.5%        100%
variant                                                           
linear_probe        72.00±1.24  74.40±2.01  85.58±0.63  89.62±0.18
global_mlp          74.20±1.59  79.50±1.05  85.00±1.09  90.88±0.88
global+local        74.05±1.19  77.60±2.29  86.67±1.46         NaN
global+desc         78.95±1.42         NaN         NaN         NaN
global+qa           78.50±1.77         NaN         NaN         NaN
full                78.90±2.21         NaN         NaN         NaN
text_only           69.40±3.71         NaN         NaN         NaN
full_shuffled_text  69.00±9.28         NaN         NaN         NaN

=== F1 (COVID full) ===
                          0.63%        1.25%        12.5%         100%
variant                                                               
linear_probe        0.616±0.026  0.632±0.039  0.844±0.007  0.901±0.006
global_mlp          0.705±0.032  0.767±0.013  0.848±0.011  0.920±0

## Notes for the write-up
- Full COVID dataset; fixed stratified 20% test split capped at 400 images (stratified, seed 42). State the cap.
- Low-data subsets (0.63%, 1.25%) are class-proportional random draws from a fixed 5% pool of the training set; 12.5% and 100% draw from the whole training set. Mean ± std is over independent draws (5 / 5 / 3 / 2), not 5-fold CV.
- + local is not run at 100% (patch tokens for 16,932 images would need ~5 GB RAM); the linear probe at 100% is the check against the DKA paper (94.85%).
- Captions: MedGemma 4B-it, Q8_0 GGUF via llama.cpp (see `run_info.json`), greedy, JSON schema enforced.